In [1]:
# ==========================================================
# TAGE-SC-L BRANCH PREDICTOR WITH FEDERATED LEARNING
# Based on: "TAGE-SC-L Branch Predictors Again" (CBP 2016)
# Winner of Championship Branch Prediction 2016
# ==========================================================

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from typing import List, Dict, Tuple, Optional
from collections import defaultdict
import math

# ==========================================================
# SETTINGS
# ==========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = "6_clients"
FL_ROUNDS = 50
BATCH_SIZE = 256
LR = 0.001
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================================
# TAGE-SC-L CONFIGURATION (64KB variant)
# Based on the CBP 2016 winning configuration
# ==========================================================
class TAGESCConfig:
    def __init__(self):
        # TAGE table sizes (in entries, for 64KB total)
        # Geometric progression of history lengths
        self.history_lengths = [7, 13, 25, 49, 97, 193, 385, 769]  # L1-L8
        
        # Table sizes (number of entries per table)
        # Total: 8 tables, sized for ~64KB storage
        self.table_sizes = [128, 256, 512, 1024, 2048, 4096, 8192, 16384]
        
        # Tag sizes (bits)
        self.tag_bits = [7, 7, 8, 8, 9, 9, 10, 10]
        
        # Counter bits (saturating counters)
        self.counter_bits = 3  # 0-7 range
        
        # Statistical Corrector (SC) tables
        self.sc_table_size = 4096
        self.sc_counter_bits = 7
        
        # Loop predictor
        self.loop_table_size = 1024
        self.loop_counter_bits = 12
        
        # Useful counter bits
        self.u_bits = 2  # 0-3 range
        
        # Alt prediction selection
        self.alt_threshold = 1
        
    def get_total_storage_kb(self):
        """Calculate total storage in KB"""
        total = 0
        
        # TAGE tables
        for i in range(len(self.table_sizes)):
            # Each entry: tag + counter + useful counter
            entry_bits = self.tag_bits[i] + self.counter_bits + self.u_bits
            total += self.table_sizes[i] * entry_bits
        
        # Statistical Corrector
        total += self.sc_table_size * self.sc_counter_bits
        
        # Loop predictor
        total += self.loop_table_size * self.loop_counter_bits
        
        return total / (8 * 1024)  # Convert to KB

# ==========================================================
# DATA LOADING
# ==========================================================
def load_csv(path):
    df = pd.read_csv(path)
    X = df.iloc[:, :-1].values.astype(np.int64)  # Use int64 for branch history
    y = df.iloc[:, -1].values.astype(np.int64)
    return X, y

def make_loader(X, y, shuffle=True):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

# ==========================================================
# HASH FUNCTIONS (as used in TAGE-SC-L)
# ==========================================================
class HashFunctions:
    @staticmethod
    def hash_pc(pc: int, seed: int = 0x9e3779b9) -> int:
        """Hash program counter"""
        h = pc
        h ^= h >> 16
        h *= 0x85ebca6b
        h ^= h >> 13
        h *= 0xc2b2ae35
        h ^= h >> 16
        h ^= seed
        return h & 0x7fffffff
    
    @staticmethod
    def hash_history(history: int, length: int, seed: int = 0) -> int:
        """Hash global history with given length"""
        # Fold history to reduce bits
        h = history
        if length > 0:
            mask = (1 << length) - 1
            h = (h ^ (h >> length)) & mask
        h ^= seed
        return h & 0x7fffffff
    
    @staticmethod
    def compute_index(pc: int, history: int, table_size: int, 
                      history_length: int, bank: int) -> int:
        """Compute table index with PC and history hash"""
        # Use PC hash
        pc_hash = HashFunctions.hash_pc(pc, bank)
        
        # Use history hash (only if history_length > 0)
        if history_length > 0:
            hist_hash = HashFunctions.hash_history(history, history_length, bank)
            # Combine hashes
            combined = pc_hash ^ hist_hash
        else:
            combined = pc_hash
        
        return combined % table_size
    
    @staticmethod
    def compute_tag(pc: int, history: int, tag_bits: int, 
                    history_length: int, bank: int) -> int:
        """Compute tag for verification"""
        pc_hash = HashFunctions.hash_pc(pc, bank + 0x9e3779b9)
        
        if history_length > 0:
            hist_hash = HashFunctions.hash_history(history, history_length, 
                                                   bank + 0x9e3779b9)
            combined = pc_hash ^ hist_hash
        else:
            combined = pc_hash
        
        mask = (1 << tag_bits) - 1
        return combined & mask

# ==========================================================
# TAGE TABLE ENTRY
# ==========================================================
class TAGE_Entry:
    """Single entry in TAGE table"""
    def __init__(self, tag_bits: int, counter_bits: int, u_bits: int):
        self.tag = 0
        self.counter = (1 << (counter_bits - 1))  # Weak taken (midpoint)
        self.useful = 0
        self.tag_mask = (1 << tag_bits) - 1
        self.counter_max = (1 << counter_bits) - 1
        self.u_max = (1 << u_bits) - 1
        
    def is_valid(self, tag: int) -> bool:
        """Check if entry is valid and tag matches"""
        return self.tag == tag
        
    def update_counter(self, taken: bool):
        """Update saturating counter"""
        if taken and self.counter < self.counter_max:
            self.counter += 1
        elif not taken and self.counter > 0:
            self.counter -= 1
            
    def get_prediction(self) -> bool:
        """Get prediction from counter"""
        return self.counter > (self.counter_max // 2)
    
    def update_useful(self, correct: bool):
        """Update useful counter"""
        if correct and self.useful < self.u_max:
            self.useful += 1
        elif not correct and self.useful > 0:
            self.useful -= 1

# ==========================================================
# TAGE TABLE (BANK)
# ==========================================================
class TAGE_Table:
    """Single TAGE table with given history length"""
    def __init__(self, history_length: int, table_size: int, 
                 tag_bits: int, counter_bits: int, u_bits: int, bank: int):
        self.history_length = history_length
        self.table_size = table_size
        self.tag_bits = tag_bits
        self.counter_bits = counter_bits
        self.u_bits = u_bits
        self.bank = bank
        
        # Initialize entries
        self.entries = [TAGE_Entry(tag_bits, counter_bits, u_bits) 
                       for _ in range(table_size)]
        
    def predict(self, pc: int, global_history: int) -> Tuple[bool, int]:
        """
        Make prediction for given PC and history
        Returns: (prediction, index)
        """
        idx = HashFunctions.compute_index(pc, global_history, self.table_size,
                                          self.history_length, self.bank)
        entry = self.entries[idx]
        
        # Check if entry is valid
        tag = HashFunctions.compute_tag(pc, global_history, self.tag_bits,
                                        self.history_length, self.bank)
        
        if entry.is_valid(tag):
            return entry.get_prediction(), idx
        else:
            # Use default prediction (weak taken)
            return True, idx
    
    def update(self, pc: int, global_history: int, taken: bool, 
               correct: bool, idx: int, alloc: bool = False):
        """
        Update TAGE table
        """
        tag = HashFunctions.compute_tag(pc, global_history, self.tag_bits,
                                        self.history_length, self.bank)
        entry = self.entries[idx]
        
        if alloc or entry.is_valid(tag):
            # Update entry
            entry.tag = tag
            entry.update_counter(taken)
            entry.update_useful(correct)
        elif entry.useful == 0:
            # Replace unused entry
            entry.tag = tag
            entry.counter = (1 << (self.counter_bits - 1))
            entry.useful = 0
            entry.update_counter(taken)
            entry.update_useful(correct)

# ==========================================================
# STATISTICAL CORRECTOR (SC)
# ==========================================================
class StatisticalCorrector:
    """Statistical corrector to fix TAGE biases"""
    def __init__(self, table_size: int, counter_bits: int):
        self.table_size = table_size
        self.counter_bits = counter_bits
        self.counter_max = (1 << counter_bits) - 1
        self.midpoint = 1 << (counter_bits - 1)
        
        # Initialize counters
        self.counters = [self.midpoint] * table_size
        
    def predict(self, pc: int, global_history: int) -> int:
        """
        Get correction value (-midpoint to +midpoint)
        """
        idx = HashFunctions.hash_pc(pc) ^ HashFunctions.hash_history(global_history, 0)
        idx %= self.table_size
        
        counter = self.counters[idx]
        return counter - self.midpoint
    
    def update(self, pc: int, global_history: int, correct: bool):
        """
        Update statistical corrector
        """
        idx = HashFunctions.hash_pc(pc) ^ HashFunctions.hash_history(global_history, 0)
        idx %= self.table_size
        
        if correct and self.counters[idx] < self.counter_max:
            self.counters[idx] += 1
        elif not correct and self.counters[idx] > 0:
            self.counters[idx] -= 1

# ==========================================================
# LOOP PREDICTOR
# ==========================================================
class LoopPredictor:
    """Predict loop exit branches"""
    def __init__(self, table_size: int, counter_bits: int):
        self.table_size = table_size
        self.counter_bits = counter_bits
        self.counter_max = (1 << counter_bits) - 1
        
        # Each entry: iteration count and confidence
        self.iterations = [0] * table_size
        self.confidence = [0] * table_size
        
    def predict(self, pc: int) -> Tuple[bool, bool]:
        """
        Predict loop exit
        Returns: (prediction, is_confident)
        """
        idx = pc % self.table_size
        iterations = self.iterations[idx]
        confidence = self.confidence[idx]
        
        if confidence > self.counter_max // 2:
            # Predict taken when iteration count reaches 0
            return iterations == 0, True
        return False, False
    
    def update(self, pc: int, taken: bool, iteration_count: int):
        """
        Update loop predictor
        """
        idx = pc % self.table_size
        
        if taken and iteration_count == 0:
            # Correctly predicted exit
            if self.confidence[idx] < self.counter_max:
                self.confidence[idx] += 1
        elif not taken and iteration_count > 0:
            # Correctly predicted continue
            if self.confidence[idx] < self.counter_max:
                self.confidence[idx] += 1
        elif self.confidence[idx] > 0:
            # Misprediction
            self.confidence[idx] -= 1
        
        # Update iteration count
        self.iterations[idx] = iteration_count

# ==========================================================
# MAIN TAGE-SC-L PREDICTOR
# ==========================================================
class TAGESCLPredictor:
    """
    TAGE-SC-L Branch Predictor
    Combines TAGE tables, Statistical Corrector, and Loop Predictor
    """
    def __init__(self, config: TAGESCConfig):
        self.config = config
        
        # Create TAGE tables
        self.tage_tables = []
        for i in range(len(config.history_lengths)):
            table = TAGE_Table(
                history_length=config.history_lengths[i],
                table_size=config.table_sizes[i],
                tag_bits=config.tag_bits[i],
                counter_bits=config.counter_bits,
                u_bits=config.u_bits,
                bank=i
            )
            self.tage_tables.append(table)
        
        # Statistical Corrector
        self.stat_corrector = StatisticalCorrector(
            config.sc_table_size,
            config.sc_counter_bits
        )
        
        # Loop Predictor
        self.loop_predictor = LoopPredictor(
            config.loop_table_size,
            config.loop_counter_bits
        )
        
        # Global history register
        self.global_history = 0
        self.history_mask = (1 << max(config.history_lengths)) - 1
        
        # Statistics
        self.predictions = 0
        self.mispredictions = 0
        
    def predict(self, pc: int) -> bool:
        """
        Predict branch direction
        Returns: True = taken, False = not taken
        """
        self.predictions += 1
        
        # Check loop predictor first (highest confidence)
        loop_pred, loop_confident = self.loop_predictor.predict(pc)
        if loop_confident:
            return loop_pred
        
        # Get predictions from all TAGE tables
        predictions = []
        indices = []
        
        for table in self.tage_tables:
            pred, idx = table.predict(pc, self.global_history)
            predictions.append(pred)
            indices.append(idx)
        
        # Find the longest history table with a valid prediction
        final_pred = predictions[-1]  # Default to longest history
        alt_pred = None
        
        # Use alt prediction from shorter tables if available
        for i in range(len(self.tage_tables) - 2, -1, -1):
            if self.tage_tables[i].entries[indices[i]].useful > self.config.alt_threshold:
                alt_pred = predictions[i]
                break
        
        # Apply statistical corrector
        sc_correction = self.stat_corrector.predict(pc, self.global_history)
        
        # Convert prediction to numeric for correction
        pred_value = 1 if final_pred else -1
        final_value = pred_value + sc_correction
        
        # Apply alt prediction if correction pushes over threshold
        if alt_pred is not None and abs(final_value) < 2:
            final_pred = alt_pred
        else:
            final_pred = final_value >= 0
        
        return final_pred
    
    def update(self, pc: int, taken: bool):
        """
        Update predictor with actual branch outcome
        """
        # Update TAGE tables
        for i, table in enumerate(self.tage_tables):
            pred, idx = table.predict(pc, self.global_history)
            correct = (pred == taken)
            
            # Determine if we need to allocate in this table
            alloc = False
            if i == len(self.tage_tables) - 1:  # Longest table
                alloc = True
            elif pred != taken and correct:  # Alternate prediction correct
                alloc = True
            
            table.update(pc, self.global_history, taken, correct, idx, alloc)
        
        # Update statistical corrector
        self.stat_corrector.update(pc, self.global_history, taken)
        
        # Update global history
        self.global_history = ((self.global_history << 1) | (1 if taken else 0))
        self.global_history &= self.history_mask
    
    def get_mpki(self, instructions: int) -> float:
        """Calculate MPKI (Mispredictions Per Kilo Instructions)"""
        if instructions == 0:
            return 0
        return (self.mispredictions / instructions) * 1000

# ==========================================================
# TAGE-SC-L MODEL WRAPPER FOR FEDERATED LEARNING
# ==========================================================
class TAGESCLModel(nn.Module):
    """
    Wrapper to make TAGE-SC-L compatible with PyTorch training loop
    Note: This is not a neural network; we're using PyTorch's structure
    for federated learning orchestration only
    """
    def __init__(self, config: TAGESCConfig):
        super().__init__()
        self.config = config
        self.predictor = TAGESCLPredictor(config)
        
        # For compatibility with PyTorch optimizer
        self.parameters_list = nn.ParameterList([
            nn.Parameter(torch.tensor(0.0))  # Dummy parameter
        ])
        
    def forward(self, x):
        """
        Forward pass for batch prediction
        x: batch of (pc, history) pairs
        """
        # For training, we need to process each sample
        batch_size = x.size(0)
        predictions = []
        
        for i in range(batch_size):
            # Assuming x contains PC and maybe other info
            pc = x[i, 0].item()
            pred = self.predictor.predict(pc)
            predictions.append(1.0 if pred else 0.0)
        
        return torch.tensor(predictions, device=x.device)
    
    def update(self, pc, taken):
        """Update predictor state"""
        self.predictor.update(pc, taken)
    
    def reset_state(self):
        """Reset predictor state for new program phase"""
        self.predictor = TAGESCLPredictor(self.config)
    
    def get_mpki(self, instructions):
        """Get MPKI metric"""
        return self.predictor.get_mpki(instructions)

# ==========================================================
# FEDERATED LEARNING WITH TAGE-SC-L
# ==========================================================
class FederatedTAGESCL:
    """Federated learning with TAGE-SC-L predictors"""
    
    def __init__(self, config: TAGESCConfig):
        self.config = config
        self.global_model = TAGESCLModel(config)
        
    def train_client(self, model: TAGESCLModel, data_loader: DataLoader):
        """
        Train client on their data stream
        In TAGE-SC-L, training is just updating state with actual outcomes
        """
        model.reset_state()  # Reset for new program phase
        
        for Xb, yb in data_loader:
            for i in range(Xb.size(0)):
                pc = Xb[i, 0].item()
                taken = yb[i].item()
                
                # Make prediction (for stats)
                pred = model.predictor.predict(pc)
                
                # Update with actual outcome
                model.update(pc, taken)
        
        return model
    
    def aggregate_models(self, client_models: List[TAGESCLModel]):
        """
        Aggregate client models (average states)
        For TAGE-SC-L, we average the saturating counters
        """
        # Create new aggregated model
        agg_model = TAGESCLModel(self.config)
        
        # Average each table's entries
        num_clients = len(client_models)
        
        # Average TAGE tables
        for table_idx in range(len(agg_model.predictor.tage_tables)):
            for entry_idx in range(agg_model.predictor.tage_tables[table_idx].table_size):
                # Average counters across clients
                sum_counter = 0
                sum_useful = 0
                
                for client_model in client_models:
                    entry = client_model.predictor.tage_tables[table_idx].entries[entry_idx]
                    sum_counter += entry.counter
                    sum_useful += entry.useful
                
                # Set averaged values
                agg_entry = agg_model.predictor.tage_tables[table_idx].entries[entry_idx]
                agg_entry.counter = sum_counter // num_clients
                agg_entry.useful = sum_useful // num_clients
        
        # Average statistical corrector
        for i in range(len(agg_model.predictor.stat_corrector.counters)):
            sum_counter = 0
            for client_model in client_models:
                sum_counter += client_model.predictor.stat_corrector.counters[i]
            agg_model.predictor.stat_corrector.counters[i] = sum_counter // num_clients
        
        # Average loop predictor
        for i in range(len(agg_model.predictor.loop_predictor.iterations)):
            sum_iter = 0
            sum_conf = 0
            for client_model in client_models:
                sum_iter += client_model.predictor.loop_predictor.iterations[i]
                sum_conf += client_model.predictor.loop_predictor.confidence[i]
            agg_model.predictor.loop_predictor.iterations[i] = sum_iter // num_clients
            agg_model.predictor.loop_predictor.confidence[i] = sum_conf // num_clients
        
        return agg_model

# ==========================================================
# EVALUATION FUNCTIONS
# ==========================================================
def evaluate_predictor(predictor: TAGESCLPredictor, data_loader: DataLoader) -> float:
    """Evaluate predictor accuracy on dataset"""
    correct = 0
    total = 0
    mispredictions = 0
    
    # Reset predictor for evaluation
    eval_predictor = TAGESCLPredictor(predictor.config)
    
    for Xb, yb in data_loader:
        for i in range(Xb.size(0)):
            pc = Xb[i, 0].item()
            actual = yb[i].item()
            
            # Predict
            pred = eval_predictor.predict(pc)
            
            # Update with actual
            eval_predictor.update(pc, actual)
            
            # Count accuracy
            if pred == actual:
                correct += 1
            else:
                mispredictions += 1
            total += 1
    
    accuracy = (correct / total) * 100 if total > 0 else 0
    mpki = (mispredictions / total) * 1000 if total > 0 else 0
    
    return accuracy, mpki

def accuracy_from_predictions(predictions, labels):
    """Calculate accuracy from predictions array"""
    return np.mean(predictions == labels) * 100

# ==========================================================
# LOAD CLIENT DATA
# ==========================================================
train_files = sorted([f for f in os.listdir(DATA_DIR) if "train" in f])

client_train_loaders = []
client_test_loaders = []
client_test_labels = []

for f in train_files:
    X_train, y_train = load_csv(os.path.join(DATA_DIR, f))
    X_test, y_test = load_csv(os.path.join(DATA_DIR, f.replace("train", "test")))
    
    client_train_loaders.append(make_loader(X_train, y_train, shuffle=True))
    client_test_loaders.append(make_loader(X_test, y_test, shuffle=False))
    client_test_labels.append(y_test)

# Global test
X_global, y_global = load_csv(os.path.join(DATA_DIR, "test.csv"))
global_loader = make_loader(X_global, y_global, shuffle=False)

num_clients = len(client_train_loaders)

# ==========================================================
# INITIALIZE TAGE-SC-L PREDICTOR
# ==========================================================
config = TAGESCConfig()
print(f"\n=== TAGE-SC-L Configuration ===")
print(f"Total storage: {config.get_total_storage_kb():.2f} KB")
print(f"Number of TAGE tables: {len(config.history_lengths)}")
print(f"History lengths: {config.history_lengths}")
print(f"Table sizes: {config.table_sizes}")
print(f"Statistical Corrector size: {config.sc_table_size} entries")
print(f"Loop Predictor size: {config.loop_table_size} entries")

federated = FederatedTAGESCL(config)

# ==========================================================
# FEDERATED LEARNING LOOP
# ==========================================================
print(f"\n{'='*60}")
print(f"Starting Federated Learning with TAGE-SC-L")
print(f"{'='*60}")

client_accuracies = []
global_accuracies = []

for rnd in range(1, FL_ROUNDS + 1):
    print(f"\n{'='*40}")
    print(f"ROUND {rnd}/{FL_ROUNDS}")
    print(f"{'='*40}")
    
    client_models = []
    client_mpki_list = []
    
    # -------------------
    # CLIENT TRAINING
    # -------------------
    for c in range(num_clients):
        # Create client model
        client_model = TAGESCLModel(config)
        
        # Train on client data stream
        trained_model = federated.train_client(client_model, client_train_loaders[c])
        
        # Evaluate on client test set
        accuracy, mpki = evaluate_predictor(trained_model.predictor, client_test_loaders[c])
        client_mpki_list.append(mpki)
        
        client_models.append(trained_model)
        
        print(f"Client {c:2d} | Test Acc: {accuracy:.2f}% | MPKI: {mpki:.2f}")
    
    # -------------------
    # AVERAGE CLIENT STATS
    # -------------------
    avg_mpki = np.mean(client_mpki_list)
    print(f"\nAvg Client MPKI: {avg_mpki:.2f}")
    
    # -------------------
    # FEDERATED AGGREGATION
    # -------------------
    global_model = federated.aggregate_models(client_models)
    
    # -------------------
    # GLOBAL TEST EVALUATION
    # -------------------
    global_accuracy, global_mpki = evaluate_predictor(global_model.predictor, global_loader)
    global_accuracies.append(global_accuracy)
    
    print(f"\n>>> Global Model | Acc: {global_accuracy:.2f}% | MPKI: {global_mpki:.2f}")
    
    # Update global model for next round
    federated.global_model = global_model

# ==========================================================
# FINAL EVALUATION
# ==========================================================
print(f"\n{'='*60}")
print(f"FINAL EVALUATION")
print(f"{'='*60}")

# Test on all client test sets with final global model
print("\nPer-client evaluation with final global model:")
client_final_accs = []
client_final_mpki = []

for c in range(num_clients):
    accuracy, mpki = evaluate_predictor(global_model.predictor, client_test_loaders[c])
    client_final_accs.append(accuracy)
    client_final_mpki.append(mpki)
    print(f"Client {c:2d} | Acc: {accuracy:.2f}% | MPKI: {mpki:.2f}")

print(f"\nAverage Client Test Accuracy: {np.mean(client_final_accs):.2f}%")
print(f"Average Client MPKI: {np.mean(client_final_mpki):.2f}")
print(f"Std Dev MPKI: {np.std(client_final_mpki):.2f}")

# Global test set evaluation
final_accuracy, final_mpki = evaluate_predictor(global_model.predictor, global_loader)
print(f"\nFINAL GLOBAL RESULTS:")
print(f"  Accuracy: {final_accuracy:.2f}%")
print(f"  MPKI: {final_mpki:.2f}")

# Detailed statistics
total_samples = len(y_global)
correct = int(final_accuracy * total_samples / 100)
print(f"\nDetailed Statistics:")
print(f"  Total samples: {total_samples}")
print(f"  Correct predictions: {correct}")
print(f"  Mispredictions: {total_samples - correct}")

# ==========================================================
# PERFORMANCE METRICS
# ==========================================================
print(f"\n{'='*60}")
print(f"PERFORMANCE SUMMARY")
print(f"{'='*60}")

# Compare with ideal predictor (perfect prediction)
print(f"\nTAGE-SC-L Metrics:")
print(f"  Storage: {config.get_total_storage_kb():.2f} KB")
print(f"  Prediction Latency: 4 cycles (as per paper)")
print(f"  Tables: {len(config.history_lengths)} TAGE tables")
print(f"  Statistical Corrector: Yes (SC)")
print(f"  Loop Predictor: Yes")
print(f"  Global History Bits: {max(config.history_lengths)}")

print(f"\nAccuracy Metrics:")
print(f"  Final Global Accuracy: {final_accuracy:.2f}%")
print(f"  Final Global MPKI: {final_mpki:.2f}")
print(f"  Average Client Accuracy: {np.mean(client_final_accs):.2f}%")
print(f"  Average Client MPKI: {np.mean(client_final_mpki):.2f}")

# ==========================================================
# SAVE MODEL (OPTIONAL)
# ==========================================================
model_path = "tagescl_final.pth"
torch.save({
    'config': {
        'history_lengths': config.history_lengths,
        'table_sizes': config.table_sizes,
        'tag_bits': config.tag_bits,
        'counter_bits': config.counter_bits,
        'u_bits': config.u_bits,
        'sc_table_size': config.sc_table_size,
        'loop_table_size': config.loop_table_size
    },
    'global_history': global_model.predictor.global_history,
    'final_accuracy': final_accuracy,
    'final_mpki': final_mpki
}, model_path)
print(f"\nModel saved to: {model_path}")

print(f"\n{'='*60}")
print(f"Training Complete!")
print(f"{'='*60}")


=== TAGE-SC-L Configuration ===
Total storage: 63.50 KB
Number of TAGE tables: 8
History lengths: [7, 13, 25, 49, 97, 193, 385, 769]
Table sizes: [128, 256, 512, 1024, 2048, 4096, 8192, 16384]
Statistical Corrector size: 4096 entries
Loop Predictor size: 1024 entries

Starting Federated Learning with TAGE-SC-L

ROUND 1/50
Client  0 | Test Acc: 79.10% | MPKI: 209.01
Client  1 | Test Acc: 83.46% | MPKI: 165.39
Client  2 | Test Acc: 73.48% | MPKI: 265.20
Client  3 | Test Acc: 50.52% | MPKI: 494.85
Client  4 | Test Acc: 66.02% | MPKI: 339.84
Client  5 | Test Acc: 58.85% | MPKI: 411.46

Avg Client MPKI: 314.29

>>> Global Model | Acc: 71.07% | MPKI: 289.25

ROUND 2/50
Client  0 | Test Acc: 79.10% | MPKI: 209.01
Client  1 | Test Acc: 83.46% | MPKI: 165.39
Client  2 | Test Acc: 73.48% | MPKI: 265.20
Client  3 | Test Acc: 50.52% | MPKI: 494.85
Client  4 | Test Acc: 66.02% | MPKI: 339.84
Client  5 | Test Acc: 58.85% | MPKI: 411.46

Avg Client MPKI: 314.29

>>> Global Model | Acc: 71.07% | MPKI

KeyboardInterrupt: 